# Vertex AI Online Inference Testing

This notebook demonstrates how to invoke our deployed **Scikit-learn classification model** hosted on a **Google Cloud Vertex AI Endpoint**. 

It covers:
1. Loading environment configuration securely.
2. Initializing the Vertex AI SDK.
3. Retrieving the live endpoint dynamically.
4. Formatting a JSON payload representing a customer transaction.
5. Sending the request and handling the returned `customer_segment` prediction.

In [ ]:
# Install necessary libraries if you haven't already
# !pip install google-cloud-aiplatform python-dotenv

### Step 1: Load Configuration and Initialize SDK

In [ ]:
import config
from google.cloud import aiplatform

# Output variables to ensure they loaded correctly
print(f"Project ID: {config.GCP_PROJECT_ID}")
print(f"Region: {config.GCP_REGION}")
print(f"Endpoint Name: {config.ENDPOINT_DISPLAY_NAME}")

# Initialize the Vertex AI SDK
aiplatform.init(project=config.GCP_PROJECT_ID, location=config.GCP_REGION)
print("\nVertex AI initialized successfully.")

### Step 2: Retrieve the Active Endpoint
We need to fetch the active endpoint object dynamically using the Display Name deployed previously.

In [ ]:
# Retrieve endpoint by Display Name
endpoints = aiplatform.Endpoint.list(
    filter=f'display_name="{config.ENDPOINT_DISPLAY_NAME}"',
    order_by="create_time desc"
)

if not endpoints:
    raise ValueError(f"Endpoint '{config.ENDPOINT_DISPLAY_NAME}' not found. Please ensure it is deployed.")

# Grab the most recent endpoint matching the name
endpoint = endpoints[0]
print(f"Endpoint retrieved successfully: {endpoint.resource_name}")

### Step 3: Define Sample Payload & Run Inference
The model expects a structured JSON array exactly matching the expected dataset features. Our serialized model pipeline (`ColumnTransformer` + Evaluator) will automatically handle the scaling and OneHot Encoding operations.

In [ ]:
# Define the sample transaction mirror dataset
# These match the features expected by the deployed model
instances = [
    {
        "category": "Electronics",
        "price": 195.50,
        "quantity": 2,
        "payment_method": "Credit Card",
        "device_type": "Mobile",
        "channel": "Social Media"
    }
]

# Send prediction request to Vertex AI Endpoint
print("Sending prediction request...")
try:
    prediction = endpoint.predict(instances=instances)
    
    # Extract and display the resulting classification segment
    print("\n--- Inference Complete ---")
    print(f"Predicted Customer Segment: {prediction.predictions[0]}")
    
except Exception as e:
    print(f"Error during prediction: {e}")